In [1]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama sentence-transformers faiss-cpu numpy docling pymupdf psycopg2-binary

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess, time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("✅ Ollama server started")

!ollama pull cogito
print("✅ cogito ready")

✅ Ollama server started

✅ cogito ready


In [3]:
import os, re, faiss, numpy as np, logging
import phonenumbers
from phonenumbers import NumberParseException

logging.getLogger("docling").setLevel(logging.ERROR)
logging.getLogger("pdfminer").setLevel(logging.ERROR)

os.system("pip install docling pymupdf -q")

from docling.document_converter import DocumentConverter
from sentence_transformers import SentenceTransformer

CHUNK_SIZE    = 400
CHUNK_OVERLAP = 80
TOP_K_CHUNKS  = 2
EMBED_MODEL   = "all-MiniLM-L6-v2"

KNOWLEDGE_FILES = [
    "/content/sample_data/hivestaff_docs.pdf"
]

converter    = DocumentConverter()
all_raw_text = {}

print(f"📄 Converting {len(KNOWLEDGE_FILES)} files with Docling...\n")

for path in KNOWLEDGE_FILES:
    if not os.path.exists(path):
        print(f"   ❌ File not found: {path}")
        continue
    try:
        print(f"   🔄 {path}")
        result = converter.convert(path)
        md     = result.document.export_to_markdown()
        if md.strip():
            all_raw_text[path] = md
            print(f"      ✅ Docling: {len(md):,} chars")
        else:
            print(f"      ⚠️  Docling empty — using pymupdf fallback...")
            import fitz
            doc  = fitz.open(path)
            text = "\n".join(p.get_text() for p in doc)
            doc.close()
            if text.strip():
                all_raw_text[path] = text
                print(f"      ✅ pymupdf fallback: {len(text):,} chars")
            else:
                print(f"      ❌ Both methods gave empty output for: {path}")
    except Exception as e:
        print(f"      ❌ Error: {e}")
        try:
            import fitz
            doc  = fitz.open(path)
            text = "\n".join(p.get_text() for p in doc)
            doc.close()
            if text.strip():
                all_raw_text[path] = text
                print(f"      ✅ pymupdf fallback: {len(text):,} chars")
        except Exception as e2:
            print(f"      ❌ Fallback also failed: {e2}")

if not all_raw_text:
    raise RuntimeError(
        "❌ No content loaded.\n"
        "   Make sure hivestaff_docs.pdf is uploaded to Colab at:\n"
        "   /content/sample_data/hivestaff_docs.pdf"
    )

os.makedirs("/content/docs_markdown", exist_ok=True)
for i, (source, md) in enumerate(all_raw_text.items()):
    fname = f"/content/docs_markdown/doc_{i+1:02d}.md"
    with open(fname, "w") as f:
        f.write(f"# Source: {source}\n\n{md}")
print(f"\n💾 Markdown saved to /content/docs_markdown/")

def chunk_text(text, source, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start  = 0
    while start < len(text):
        end   = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append({"text": chunk, "source": source})
        start += chunk_size - overlap
    return chunks

print(f"\n📐 Chunking {len(all_raw_text)} documents...")
ALL_CHUNKS = []
for source, text in all_raw_text.items():
    chunks = chunk_text(text, source)
    ALL_CHUNKS.extend(chunks)
    print(f"   {os.path.basename(source)}: {len(chunks)} chunks")

print(f"\n   Total chunks: {len(ALL_CHUNKS)}")

print(f"\n🧠 Loading embedding model: {EMBED_MODEL} ...")
EMBED_MDL = SentenceTransformer(EMBED_MODEL)

print(f"⚙️  Embedding {len(ALL_CHUNKS)} chunks...")
texts      = [c["text"] for c in ALL_CHUNKS]
embeddings = EMBED_MDL.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

dim         = embeddings.shape[1]
FAISS_INDEX = faiss.IndexFlatIP(dim)
FAISS_INDEX.add(embeddings)

test_vec = EMBED_MDL.encode(["staff management"], convert_to_numpy=True).astype("float32")
faiss.normalize_L2(test_vec)
scores, idxs = FAISS_INDEX.search(test_vec, 1)
if idxs[0][0] == -1:
    print("⚠️  WARNING: Index empty!")
else:
    print(f"\n   ✅ Index working — score: {scores[0][0]:.3f}")
    print(f"   Sample: {ALL_CHUNKS[idxs[0][0]]['text'][:100]}...")

print(f"""
🎉 RAG system ready!
   Files   : {len(all_raw_text)}
   Chunks  : {len(ALL_CHUNKS)}
   Vectors : {FAISS_INDEX.ntotal}
""")

📄 Converting 1 files with Docling...

   🔄 /content/sample_data/hivestaff_docs.pdf


[INFO] 2026-05-13 04:54:00,049 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-13 04:54:00,059 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-13 04:54:00,127 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-13 04:54:00,128 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-13 04:54:00,695 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-13 04:54:00,696 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-13 04:54:00,703 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-13 04:54:00,704 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-05-13 04:54:30,160 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-13 04:55:02,315 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-13 04:55:04,775 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-13 04:55:05,730 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-13 04:56:00,435 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-05-13 04:56:01,527 [RapidOCR] main.py:132: The text detection result is empty


      ✅ Docling: 59,004 chars

💾 Markdown saved to /content/docs_markdown/

📐 Chunking 1 documents...
   hivestaff_docs.pdf: 185 chunks

   Total chunks: 185

🧠 Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⚙️  Embedding 185 chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


   ✅ Index working — score: 0.571
   Sample: ## Shift   Roster

Create and manage employee shift schedules efficiently

<!-- image -->

## Leave
...

🎉 RAG system ready!
   Files   : 1
   Chunks  : 185
   Vectors : 185



In [ ]:
import psycopg2

NEON_URL = "...."
try:
    conn = psycopg2.connect(NEON_URL)
    cur  = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS leads (
            id            SERIAL PRIMARY KEY,
            timestamp     TIMESTAMPTZ DEFAULT NOW(),
            score         INT,
            reason        TEXT,
            name          TEXT,
            company       TEXT,
            email         TEXT UNIQUE,
            phone         TEXT UNIQUE,
            demo_datetime TEXT
        )
    """)
    # If table already existed without phone UNIQUE, add it safely
    cur.execute("""
        DO $$
        BEGIN
            IF NOT EXISTS (
                SELECT 1 FROM pg_constraint
                WHERE conname = 'leads_phone_unique'
            ) THEN
                ALTER TABLE leads ADD CONSTRAINT leads_phone_unique UNIQUE (phone);
            END IF;
        END $$;
    """)
    # If table already existed without demo_datetime column, add it safely
    cur.execute("""
        DO $$
        BEGIN
            IF NOT EXISTS (
                SELECT 1 FROM information_schema.columns
                WHERE table_name = 'leads' AND column_name = 'demo_datetime'
            ) THEN
                ALTER TABLE leads ADD COLUMN demo_datetime TEXT;
            END IF;
        END $$;
    """)
    conn.commit()
    cur.execute("SELECT COUNT(*) FROM leads")
    count = cur.fetchone()[0]
    cur.close()
    conn.close()
    print("✅ Connected to Neon PostgreSQL!")
    print(f"✅ leads table ready — {count} existing leads in DB")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to Neon PostgreSQL!
✅ leads table ready — 3 existing leads in DB


In [9]:
import json, os, ollama, psycopg2
from datetime import datetime
from zoneinfo import ZoneInfo

MODEL = "cogito"
IST   = ZoneInfo("Asia/Kolkata")  # Indian timezone

DOC_BASE = "https://document.hivestaff.in"
URL_MAP = {
    "overview"       : f"{DOC_BASE}",
    "login"          : f"{DOC_BASE}/getting-started",
    "dashboard"      : f"{DOC_BASE}/getting-started",
    "password"       : f"{DOC_BASE}/getting-started",
    "role"           : f"{DOC_BASE}/getting-started",
    "permission"     : f"{DOC_BASE}/getting-started",
    "configuration"  : f"{DOC_BASE}/getting-started",
    "human resource" : f"{DOC_BASE}/modules/human-resource",
    "hr"             : f"{DOC_BASE}/modules/human-resource",
    "employee"       : f"{DOC_BASE}/modules/human-resource",
    "attendance"     : f"{DOC_BASE}/modules/human-resource",
    "leave"          : f"{DOC_BASE}/modules/human-resource",
    "payroll"        : f"{DOC_BASE}/modules/human-resource",
    "salary"         : f"{DOC_BASE}/modules/human-resource",
    "loan"           : f"{DOC_BASE}/modules/human-resource",
    "performance"    : f"{DOC_BASE}/modules/human-resource",
    "recruitment"    : f"{DOC_BASE}/modules/recruitment",
    "hiring"         : f"{DOC_BASE}/modules/recruitment",
    "candidate"      : f"{DOC_BASE}/modules/recruitment",
    "interview"      : f"{DOC_BASE}/modules/recruitment",
    "job"            : f"{DOC_BASE}/modules/recruitment",
    "letter"         : f"{DOC_BASE}/modules/letter-generate",
    "template"       : f"{DOC_BASE}/modules/letter-generate",
    "asset"          : f"{DOC_BASE}/modules/assets",
    "inventory"      : f"{DOC_BASE}/modules/assets",
    "notice"         : f"{DOC_BASE}/modules/notice-board",
    "announcement"   : f"{DOC_BASE}/modules/notice-board",
    "reception"      : f"{DOC_BASE}/modules/reception",
    "visitor"        : f"{DOC_BASE}/modules/reception",
    "lead"           : f"{DOC_BASE}/modules/leads",
    "crm"            : f"{DOC_BASE}/modules/leads",
    "client"         : f"{DOC_BASE}/modules/leads",
    "contractor"     : f"{DOC_BASE}/modules/contractor",
    "vendor"         : f"{DOC_BASE}/modules/contractor",
    "construction"   : f"{DOC_BASE}/modules/work-construction",
    "project"        : f"{DOC_BASE}/modules/work-construction",
    "purchase"       : f"{DOC_BASE}/modules/purchase",
    "procurement"    : f"{DOC_BASE}/modules/purchase",
    "quotation"      : f"{DOC_BASE}/modules/purchase",
    "material"       : f"{DOC_BASE}/modules/materials",
    "stock"          : f"{DOC_BASE}/modules/materials",
    "warehouse"      : f"{DOC_BASE}/modules/warehouse",
    "company"        : f"{DOC_BASE}/modules/company",
    "branch"         : f"{DOC_BASE}/modules/company",
}

def get_doc_link(query):
    q = query.lower()
    for keyword, url in URL_MAP.items():
        if keyword in q:
            return url
    return None

def retrieve(query):
    vec = EMBED_MDL.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(vec)
    scores, idxs = FAISS_INDEX.search(vec, TOP_K_CHUNKS)
    context = "\n\n".join(ALL_CHUNKS[i]["text"] for i in idxs[0] if i != -1)
    link    = get_doc_link(query)
    return context, link

BASE_SYSTEM = """You are Hive, a smart and friendly customer service AI for HiveStaff ERP.

Your objectives:
1. Greet warmly and understand the customer's business challenges
2. Match their pain points to relevant HiveStaff features
3. Build genuine interest and convince them to book a FREE DEMO
4. Once interested, collect: Full Name, Company Name, Email, Phone Number
5. After contact details are collected, ask: "What date and time works best for your demo? (Please share date and time in IST)"

Rules:
- ONLY talk about HiveStaff. Politely redirect off-topic questions.
- For pricing: "Our team will share a tailored quote during your free demo"
- When user shows interest say: "To schedule your free demo, could I get your name, company, email and phone?"
- After contact details collected, always ask for preferred demo date and time
- Be warm and conversational — not robotic or pushy
- Answer ONLY using the context provided. Never fabricate features.
- NEVER add any links or URLs yourself. Links are handled externally.
"""

SENTIMENT_PROMPT = """Analyze this customer conversation strictly and objectively. Return ONLY valid JSON, no extra text:
{
  "interested": true or false,
  "score": 0-10,
  "reason": "one line explanation",
  "lead": { "name": null, "company": null, "email": null, "phone": null, "demo_datetime": null }
}

Scoring rules — follow these exactly:
- 0-2  : Just saying hi, no interest, off-topic, one word replies
- 3-4  : Asking very basic generic questions, no specific interest shown
- 5    : Mildly curious, asked 1-2 features but no demo interest shown
- 6-7  : Clearly interested, asked multiple questions OR mentioned pain points
- 8    : Specifically asked for demo OR gave contact details
- 9    : Asked for demo AND gave contact details
- 10   : Asked for demo, gave all contact details AND confirmed a demo time

Rules:
- interested=true ONLY if score >= 6
- Do NOT give 8+ just because someone gave contact info without genuine interest
- Do NOT round up — be strict and realistic
- Extract any contact details the user mentioned
- Extract demo date/time if mentioned (e.g. "15 May 2026 3:00 PM IST")
- Score based on: depth of questions, specific pain points, demo intent, engagement quality"""

# ── EMAIL VALIDATION ──────────────────────────────────────────────────────────

def validate_email_format(email: str) -> bool:
    pattern = r'^[a-zA-Z0-9._%+\-]{1,64}@[a-zA-Z0-9.\-]{1,255}\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email.strip()))

def validate_email_domain(email: str) -> tuple[bool, str]:
    try:
        import dns.resolver
        domain = email.split('@')[1].strip()
        try:
            dns.resolver.resolve(domain, 'MX')
            return True, "MX record found"
        except (dns.resolver.NoAnswer, dns.resolver.NXDOMAIN):
            pass
        try:
            dns.resolver.resolve(domain, 'A')
            return True, "A record found"
        except (dns.resolver.NoAnswer, dns.resolver.NXDOMAIN):
            pass
        return False, f"Domain '{domain}' does not exist or has no mail records"
    except Exception as e:
        print(f"⚠️  DNS check skipped: {e}")
        return True, "DNS check skipped"

def validate_email(email: str) -> tuple[bool, str]:
    email = email.strip()
    if not validate_email_format(email):
        return False, (
            f"'{email}' is not a valid email address. "
            "Please enter a proper email like john@company.com"
        )
    domain_ok, reason = validate_email_domain(email)
    if not domain_ok:
        return False, (
            f"The email domain seems invalid ({reason}). "
            "Please enter a real business or personal email."
        )
    return True, "valid"

# ── PHONE VALIDATION ──────────────────────────────────────────────────────────

FAKE_PHONE_PATTERNS = [
    r'^(\d)\1{6,}$',
    r'^1234567890$',
    r'^0123456789$',
    r'^9876543210$',
    r'^(\d{1,3})\1{2,}$',
    r'^12345678\d{2}$',
    r'^0{7,}',
]

def is_fake_phone_pattern(digits: str) -> bool:
    for pattern in FAKE_PHONE_PATTERNS:
        if re.match(pattern, digits):
            return True
    return False

def validate_phone(phone: str, default_region: str = "IN") -> tuple[bool, str]:
    phone   = phone.strip()
    cleaned = re.sub(r'[\s\-\(\)\.]+', '', phone)
    digits  = re.sub(r'\D', '', cleaned)

    if len(digits) < 7 or len(digits) > 15:
        return False, "Phone number should be between 7 and 15 digits. Please re-enter."

    if is_fake_phone_pattern(digits):
        return False, (
            f"'{phone}' looks like a placeholder number. "
            "Please enter your actual phone number."
        )
    try:
        parsed = phonenumbers.parse(phone, default_region)
        if not phonenumbers.is_valid_number(parsed):
            return False, "That phone number doesn't appear valid for its region. Please re-enter."
        if not phonenumbers.is_possible_number(parsed):
            return False, "That phone number length seems off. Try with country code e.g. +91 98765 43210."
        return True, "valid"
    except NumberParseException:
        return False, "Couldn't parse that number. Please enter with country code e.g. +91 98765 43210."

# ── EXTRACT FROM TEXT ─────────────────────────────────────────────────────────

def extract_email_from_text(text: str):
    match = re.search(
        r'\b[a-zA-Z0-9._%+\-]{1,64}@[a-zA-Z0-9.\-]{1,255}\.[a-zA-Z]{2,}\b',
        text
    )
    return match.group(0) if match else None

def extract_phone_from_text(text: str):
    match = re.search(r'(\+?\d[\d\s\-\(\)\.]{6,}[\d])', text)
    return match.group(0).strip() if match else None

def extract_demo_datetime_from_text(text: str):
    """
    Extract demo date/time mentioned by user.
    Looks for patterns like: '15 May 3pm', 'tomorrow 10am', '20/05 at 2:30 PM' etc.
    Returns raw string — stored as-is, AI already formats it nicely.
    """
    patterns = [
        r'\b\d{1,2}[\/\-]\d{1,2}(?:[\/\-]\d{2,4})?\s+(?:at\s+)?\d{1,2}(?::\d{2})?\s*(?:am|pm|AM|PM)\b',
        r'\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\w*\s+\d{1,2}(?:st|nd|rd|th)?\s*(?:,?\s*\d{4})?\s+(?:at\s+)?\d{1,2}(?::\d{2})?\s*(?:am|pm|AM|PM)\b',
        r'\b\d{1,2}(?:st|nd|rd|th)?\s+(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\w*\s*(?:,?\s*\d{4})?\s+(?:at\s+)?\d{1,2}(?::\d{2})?\s*(?:am|pm|AM|PM)\b',
        r'\b(?:tomorrow|today|monday|tuesday|wednesday|thursday|friday|saturday|sunday)\s+(?:at\s+)?\d{1,2}(?::\d{2})?\s*(?:am|pm|AM|PM)\b',
        r'\b\d{1,2}(?::\d{2})?\s*(?:am|pm|AM|PM)\s+(?:on\s+)?\d{1,2}[\/\-]\d{1,2}(?:[\/\-]\d{2,4})?\b',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(0).strip()
    return None

# ── DUPLICATE CHECK ───────────────────────────────────────────────────────────

def check_duplicate_in_db(email: str = None, phone: str = None) -> tuple[bool, str]:
    try:
        conn = psycopg2.connect(NEON_URL)
        cur  = conn.cursor()
        if email:
            cur.execute("SELECT id FROM leads WHERE email = %s", (email.strip(),))
            if cur.fetchone():
                cur.close(); conn.close()
                return True, "email"
        if phone:
            cur.execute("SELECT id FROM leads WHERE phone = %s", (phone.strip(),))
            if cur.fetchone():
                cur.close(); conn.close()
                return True, "phone"
        cur.close(); conn.close()
        return False, None
    except Exception as e:
        print(f"⚠️  Duplicate check error: {e}")
        return False, None

# ── SAVE LEAD ─────────────────────────────────────────────────────────────────

def save_lead_to_db(score, reason, lead, demo_datetime=None):
    try:
        conn = psycopg2.connect(NEON_URL)
        cur  = conn.cursor()
        cur.execute("""
            INSERT INTO leads (timestamp, score, reason, name, company, email, phone, demo_datetime)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (email) DO NOTHING
        """, (
            datetime.now(IST),          # ← IST timestamp
            score,
            reason,
            lead.get("name"),
            lead.get("company"),
            lead.get("email"),
            lead.get("phone"),
            demo_datetime,
        ))
        conn.commit()
        cur.close()
        conn.close()
        return True
    except Exception as e:
        print(f"❌ DB error: {e}")
        return False

def analyze_and_save(history, demo_datetime=None):
    convo = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history)
    resp  = ollama.chat(model=MODEL, messages=[
        {"role": "system", "content": SENTIMENT_PROMPT},
        {"role": "user",   "content": convo}
    ])
    try:
        text = resp['message']['content']
        data = json.loads(text[text.find('{'):text.rfind('}')+1])
    except:
        print("⚠️  Could not parse sentiment."); return

    print(f"\n📊 Interest Score: {data['score']}/10 — {data['reason']}")

    if data.get("interested"):
        lead = data["lead"]
        # Use collected demo_datetime from chat() if sentiment didn't extract it
        final_demo_dt = demo_datetime or lead.get("demo_datetime")
        saved = save_lead_to_db(data["score"], data["reason"], lead, final_demo_dt)
        if saved:
            print("✅ Lead saved to Neon PostgreSQL!")
            print(json.dumps({
                "timestamp"    : datetime.now(IST).strftime("%d-%m-%Y %I:%M %p IST"),
                "score"        : data["score"],
                "reason"       : data["reason"],
                "demo_datetime": final_demo_dt,
                **{k: v for k, v in lead.items() if v and k != "demo_datetime"}
            }, indent=2))
    else:
        print("❌ Not interested — lead skipped.")

# ── CHAT ──────────────────────────────────────────────────────────────────────

def chat():
    history          = []
    collected        = {"name": None, "company": None, "email": None, "phone": None, "demo_datetime": None}
    pending_validate = None

    print("=" * 55)
    print("  🐝 HiveStaff AI Assistant  |  type 'exit' to quit")
    print("=" * 55)

    greeting_resp = ollama.chat(model=MODEL, messages=[
        {
            "role"   : "system",
            "content": BASE_SYSTEM + (
                "\n\nGreet the customer warmly, introduce yourself as Hive from "
                "HiveStaff ERP, and ask about their business challenges in 2-3 sentences."
            )
        }
    ])
    greeting = greeting_resp['message']['content']
    print(f"\n🐝 Hive: {greeting}\n")
    history.append({"role": "assistant", "content": greeting})

    while True:
        user = input("You: ").strip()
        if not user:
            continue
        if user.lower() == "exit":
            break

        found_email      = extract_email_from_text(user)
        found_phone      = extract_phone_from_text(user)
        found_demo_dt    = extract_demo_datetime_from_text(user)
        validation_error = None
        duplicate_field  = None

        # ── Email: format → domain → duplicate ───────────────────────────────
        if found_email and not collected["email"]:
            ok, reason = validate_email(found_email)
            if not ok:
                validation_error = reason
                pending_validate = "email"
                print(f"   ❌ Email rejected: {found_email} — {reason}")
            else:
                is_dup, _ = check_duplicate_in_db(email=found_email)
                if is_dup:
                    duplicate_field  = "email"
                    validation_error = (
                        f"The email '{found_email}' is already registered with us. "
                        "Please provide a different email address."
                    )
                    pending_validate = "email"
                    print(f"   ❌ Duplicate email: {found_email}")
                else:
                    collected["email"] = found_email
                    print(f"   ✅ Email validated & unique: {found_email}")

        # ── Phone: format → fake → duplicate ─────────────────────────────────
        if found_phone and not collected["phone"] and not validation_error:
            ok, reason = validate_phone(found_phone)
            if not ok:
                validation_error = reason
                pending_validate = "phone"
                print(f"   ❌ Phone rejected: {found_phone} — {reason}")
            else:
                is_dup, _ = check_duplicate_in_db(phone=found_phone)
                if is_dup:
                    duplicate_field  = "phone"
                    validation_error = (
                        f"The phone number '{found_phone}' is already registered. "
                        "Please provide a different number."
                    )
                    pending_validate = "phone"
                    print(f"   ❌ Duplicate phone: {found_phone}")
                else:
                    collected["phone"] = found_phone
                    print(f"   ✅ Phone validated & unique: {found_phone}")

        # ── Demo datetime: capture if mentioned ───────────────────────────────
        if found_demo_dt and not collected["demo_datetime"]:
            collected["demo_datetime"] = found_demo_dt
            print(f"   ✅ Demo schedule captured: {found_demo_dt}")

        # ── Block AI if invalid or duplicate ──────────────────────────────────
        if validation_error:
            if duplicate_field:
                error_context = (
                    f"The customer's {duplicate_field} already exists in our database. "
                    f"Error: {validation_error}"
                )
                ask_for = f"a different {duplicate_field}"
            else:
                error_context = f"Validation failed: {validation_error}"
                ask_for = (
                    "their correct email address"
                    if pending_validate == "email"
                    else "their correct phone number"
                )

            error_prompt = (
                BASE_SYSTEM
                + f"\n\nContact detail issue:\n{error_context}\n"
                f"Politely tell the customer exactly what is wrong and ask them to provide "
                f"{ask_for}. Do NOT proceed with scheduling or collecting other info. "
                f"Be empathetic and brief."
            )
            history.append({"role": "user", "content": user})
            err_resp = ollama.chat(model=MODEL, messages=[
                {"role": "system", "content": error_prompt}
            ] + history)
            reply = err_resp['message']['content']
            history.append({"role": "assistant", "content": reply})
            print(f"\n🐝 Hive: {reply}\n")
            continue

        # ── Normal RAG flow ───────────────────────────────────────────────────
        context, doc_link = retrieve(user)

        link_instruction = (
            f"\n\nAt the END of your response add exactly this line: '📖 Read more: {doc_link}'"
            if doc_link else ""
        )

        collected_info = "\n".join(
            f"- {k.replace('_', ' ').title()}: {v}" for k, v in collected.items() if v
        )
        collection_note = (
            f"\n\n== Already collected ==\n{collected_info}\n== End =="
            if collected_info else ""
        )

        # Tell AI whether demo schedule is still pending
        schedule_note = ""
        all_contact_done = all([collected["name"], collected["email"],
                                collected["phone"], collected["company"]])
        if all_contact_done and not collected["demo_datetime"]:
            schedule_note = (
                "\n\nAll contact details are collected. "
                "Now warmly ask the customer what date and time suits them for the demo (IST). "
                "Be specific — ask for both date and time."
            )

        system_prompt = (
            BASE_SYSTEM
            + f"\n\n== Relevant HiveStaff Info ==\n{context}\n== End =="
            + collection_note
            + schedule_note
            + link_instruction
        )

        history.append({"role": "user", "content": user})
        resp  = ollama.chat(model=MODEL, messages=[
            {"role": "system", "content": system_prompt}
        ] + history)
        reply = resp['message']['content']
        history.append({"role": "assistant", "content": reply})
        print(f"\n🐝 Hive: {reply}\n")

    if history:
        print("\n🔍 Analyzing conversation...")
        analyze_and_save(history, demo_datetime=collected["demo_datetime"])


chat()

  🐝 HiveStaff AI Assistant  |  type 'exit' to quit

🐝 Hive: 

Hello there! I'm Hive, your friendly guide to everything HiveStaff ERP has to offer. How's your business doing today? Are you facing any specific challenges with employee management, payroll processing, or administrative tasks that we might be able to help streamline for you?

You: i do it with software already

🐝 Hive: That's great! HiveStaff is designed to integrate seamlessly with existing systems. What I'm curious about is whether our unique features like centralized project management, advanced task tracking, or the automated activity log would offer any benefits in your current setup? We could also discuss how our member assignment system and quick reference guide might help streamline certain tasks for you. Would you be open to exploring these possibilities further with a free demo?

You: okkay lets see in demo

🐝 Hive: Excellent! Let me get the ball rolling for your free HiveStaff ERP demonstration. To schedule it, c